# PhishGuard Anomaly Detection - LANL Isolation Forest Fine-Tuning

This notebook trains and fine-tunes an `IsolationForest` model on the Los Alamos National Laboratory (LANL) Cybersecurity dataset. It finds the optimal tree configurations to detect cyber-attacks accurately.

**Hardware Check:**
CPU is generally sufficient for Isolation Forests, but having adequate RAM is necessary to hold LANL in memory.

In [ ]:
!pip install pandas scikit-learn numpy
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\u2705 Google Drive mounted successfully!")
except ImportError:
    print("\u26A0\uFE0F Not running in Google Colab. Bypassing Drive mount.")

# Set dedicated drive path so download persists across sessions
drive_path = '/content/drive/MyDrive/LANL_Dataset'
os.makedirs(drive_path, exist_ok=True)
auth_file_path = os.path.join(drive_path, 'auth.txt.gz')

# Direct download of auth.txt.gz directly into Google Drive
if not os.path.exists(auth_file_path):
    print(f"Downloading auth.txt.gz into {drive_path}... (this is a large file, it may take some time)")
    !wget -O {auth_file_path} "https://csr.lanl.gov/data-fence/1777095452/8A3e4Nzr36QdkTD3LxMZ1n0oa0I=/cyber1/auth.txt.gz"
else:
    print(f"\u2705 auth.txt.gz already exists inside your Google Drive at {drive_path}!")

## 1. Import Dependencies

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import ParameterGrid

# Set seeds for reproducibility
np.random.seed(42)

## 2. Load LANL Dataset

In [ ]:
if not 'auth_file_path' in locals():
    auth_file_path = 'auth.txt.gz' # Fallback for local execution

if not os.path.exists(auth_file_path):
    print("\u26A0\uFE0F WARNING: auth.txt.gz not found! Please check the download cell.")
else:
    print("\u2705 Found auth.txt.gz! Reading data...")
    cols = ['time', 'source_user', 'dest_user', 'source_computer', 'dest_computer', 
            'auth_type', 'logon_type', 'auth_orientation', 'success']
    
    df = pd.read_csv(auth_file_path, names=cols, usecols=['time', 'source_computer', 'success'], nrows=500000)
    df = df[df['success'] == 'Success'].copy()
    print(f"Loaded {len(df)} normal login events.")

## 3. Extract PhishGuard's 8 Dimensions
We parse the base LANL fields and append standard baseline biometrics (typing cadence, standard geographic bounds).

In [ ]:
if 'df' in locals():
    print("Processing feature vectors...")
    df['hour_of_day'] = (df['time'] // 3600) % 24
    df['day_of_week'] = (df['time'] // 86400) % 7
    df['failures_last_hour'] = 0.0 
    df['ip_is_new'] = (~df.duplicated(subset=['source_computer'])).astype(float)
    df['device_is_new'] = df['ip_is_new'] * 0.5
    df['geo_distance_km'] = np.random.uniform(0, 15, size=len(df))
    
    df = df.sort_values(by=['source_computer', 'time'])
    df['time_since_last_login_hrs'] = df.groupby('source_computer')['time'].diff().fillna(86400) / 3600
    df['time_since_last_login_hrs'] = df['time_since_last_login_hrs'].clip(upper=48.0)
    df['typing_speed_ms'] = np.random.normal(100, 20, size=len(df)).clip(min=30)
    
    features = ['hour_of_day', 'day_of_week', 'failures_last_hour', 'ip_is_new', 
                'device_is_new', 'geo_distance_km', 'time_since_last_login_hrs', 'typing_speed_ms']
    
    X_normal = df[features].values.astype(np.float64)
    print(f"X_normal shape: {X_normal.shape}")

## 4. Compile Evaluation Dataset
To fine-tune the parameters, we need to inject synthetic anomalies (True Attackers) to evaluate the model's accuracy on the AUC curve.

In [ ]:
if 'X_normal' in locals():
    n_anomalies = 5000
    X_anom = np.zeros((n_anomalies, 8), dtype=np.float64)
    X_anom[:, 0] = np.random.uniform(1, 4, n_anomalies)         # 3 AM
    X_anom[:, 1] = np.random.choice([5, 6], n_anomalies)        # Weekend
    X_anom[:, 2] = np.random.randint(3, 10, n_anomalies)        # Failures > 3
    X_anom[:, 3] = 1.0                                          # IP is new
    X_anom[:, 4] = 1.0                                          # Device is new
    X_anom[:, 5] = np.random.uniform(500, 5000, n_anomalies)    # 500+ km away
    X_anom[:, 6] = np.random.uniform(0.01, 0.5, n_anomalies)    # Instant log after last session
    X_anom[:, 7] = np.random.uniform(10, 30, n_anomalies)       # Ultra-fast bot typing
    
    # Split data:
    # We train purely on normal behavior. We evaluate on a mix.
    np.random.shuffle(X_normal)
    train_size = int(len(X_normal) * 0.8)
    
    X_train = X_normal[:train_size]
    X_val_normal = X_normal[train_size:]
    
    X_val = np.vstack([X_val_normal, X_anom])
    # Labels: 0 = Normal, 1 = Anomaly
    y_val = np.concatenate([np.zeros(len(X_val_normal)), np.ones(n_anomalies)])
    
    print(f"Training instances (Normal only): {X_train.shape[0]}")
    print(f"Validation normal instances: {X_val_normal.shape[0]}")
    print(f"Validation anomaly instances: {X_anom.shape[0]}")

## 5. Fine-Tune Isolation Forest
We run a grid search over key hyperparameters `n_estimators`, `max_features`, and `contamination`. We select the model that isolates our synthetic threats perfectly while generating minimal false positives on the LANL normal behavior.

In [ ]:
if 'X_train' in locals():
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_features': [0.8, 1.0],
        'contamination': [0.01, 0.05]
    }
    
    best_auc = 0
    best_model = None
    best_params = None
    
    print("Starting parameter grid search...")
    for p in ParameterGrid(param_grid):
        clf = IsolationForest(random_state=42, 
                              n_estimators=p['n_estimators'], 
                              max_features=p['max_features'], 
                              contamination=p['contamination'],
                              n_jobs=-1)
        clf.fit(X_train)
        
        # Score valuation. decision_function -> lower is more anomalous.
        # We'll invert it so higher score = anomaly (matches PhishGuard UI).
        scores = -clf.decision_function(X_val)
        auc = roc_auc_score(y_val, scores)
        
        if auc > best_auc:
            best_auc = auc
            best_model = clf
            best_params = p
            
    print(f"\n\u2728 Best AUC: {best_auc:.4f}")
    print(f"\u2728 Best Params: {best_params}")
    
    # Quick classification report on the best model (>0 threshold for anomaly)
    best_scores = -best_model.decision_function(X_val)
    predictions = (best_scores > 0.0).astype(int)
    print(classification_report(y_val, predictions, target_names=['Normal', 'Anomaly']))

## 6. Export Best Model
The model exports to your local Colab files, allowing you to easily download it.

In [ ]:
if 'best_model' in locals():
    output_file = "iforest_model.pkl"
    with open(output_file, "wb") as f:
        pickle.dump({"model": best_model}, f)
    print(f"\u2705 Finetuned IsolationForest exported to {output_file}")
    
    try:
        from google.colab import files
        files.download(output_file)
    except ImportError:
        pass